# Employee Attrition & Financial Risk Analytics
## Notebook 2 of 3 — Feature Engineering, Modeling & Financial Risk

**Goal of this notebook:** encode the cleaned dataset for machine learning, train and compare a
Logistic Regression and a Random Forest classifier, then translate the winning model's
predictions into an estimated financial risk per employee and break that risk down by business
segment (department, job role, salary, overtime, age, job satisfaction).

**Input:** `data/employee_attrition_clean.csv` (produced by Notebook 1)

**Outputs of this notebook** (used by Notebook 3):
- `models/scaler.pkl`, `models/logistic_regression_model.pkl`
- `data/X_train.csv`, `data/X_test.csv`, `data/y_train.csv`, `data/y_test.csv`
- `results/predictions_with_financial_risk.csv`


## 1. Setup & Load Cleaned Data

Loading the output of Notebook 1. `SalaryGroup` and `AgeGroup` are restored as **ordered
categoricals** here — CSV round-tripping loses that ordering by default, and the segment-risk
plots later in this notebook rely on Low → Medium → High → Very High (and 18-25 → 46-60) ordering
to be meaningful.


In [ ]:
import os
import joblib
import pandas as pd

os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("visualizations", exist_ok=True)

pd.set_option('display.max_columns', None)

df_clean = pd.read_csv('data/employee_attrition_clean.csv')

# Restore the ordered-category dtype lost in the CSV round trip from Notebook 1,
# so downstream groupby/plot ordering matches the original notebook exactly.
df_clean['SalaryGroup'] = pd.Categorical(
    df_clean['SalaryGroup'],
    categories=['Low', 'Medium', 'High', 'Very High'],
    ordered=True
)
df_clean['AgeGroup'] = pd.Categorical(
    df_clean['AgeGroup'],
    categories=['18-25', '26-35', '36-45', '46-60'],
    ordered=True
)

df_clean.info()


## 2. Feature Engineering for Machine Learning

A modeling copy, `df_ml`, is created and stripped of columns that would leak the target or add no
predictive value (`Attrition` itself, and the two grouped/derived columns `SalaryGroup`/`AgeGroup`
which are redundant with the raw `MonthlyIncome`/`Age` already in the feature set). Remaining
categorical columns are then one-hot encoded.


In [ ]:
df_ml = df_clean.copy()


In [ ]:
columns_to_drop = [
    'Attrition',
    'SalaryGroup',
    'AgeGroup',
    'MonthlyIncome'
]

df_ml = df_ml.drop(
    columns = columns_to_drop
)


In [ ]:
df_ml.shape


In [ ]:
df_ml.select_dtypes(include = 'object')


In [ ]:
df_ml = pd.get_dummies(
    df_ml,
    drop_first = True,
    dtype = int
)

df_ml.head()


In [ ]:
df_ml.shape


In [ ]:
df_ml.info()


## 3. Train/Test Split & Scaling


In [ ]:
X = df_ml.drop(columns = 'AttritionNumeric')

y = df_ml['AttritionNumeric']

print("Features (X):", X.shape)
print("Target (y):", y.shape)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)



In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


**Dependency hand-off:** the raw (pre-scaling) `X_train`/`X_test`/`y_train`/`y_test` are saved
now, so Notebook 3 can rebuild SHAP-ready inputs without needing anything still in this kernel's
memory.


In [ ]:
X_train.to_csv('data/X_train.csv', index=True)
X_test.to_csv('data/X_test.csv', index=True)
y_train.to_csv('data/y_train.csv', index=True)
y_test.to_csv('data/y_test.csv', index=True)
print("Saved raw train/test splits -> data/")


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


**Dependency hand-off:** the *fitted* scaler is saved so Notebook 3 can reproduce these exact
scaled features via `scaler.transform(...)` — never a freshly-fit scaler, which would silently
give incorrect SHAP explanations.


In [ ]:
joblib.dump(scaler, 'models/scaler.pkl')
print("Saved fitted scaler -> models/scaler.pkl")


## 4. Model 1 — Logistic Regression


In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter = 1000)

model.fit(X_train_scaled, y_train)

model


In [ ]:
y_pred = model.predict(X_test_scaled)

y_pred


In [ ]:
y_prob = model.predict_proba(X_test_scaled)
y_prob[:5]


In [ ]:
predictions = X_test.copy()

predictions['Actual'] = y_test.values
predictions['Predicted'] = y_pred
predictions['Probability_of_Leaving'] = y_prob[:,1]

predictions.head()


In [ ]:
predictions.to_csv(
    'results/logistic_regression_predictions.csv',
    index=False
)


In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test,y_pred)

print("Accuracy =", accuracy)


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test,y_pred)

print(cm)


In [ ]:
from sklearn.metrics import classification_report

print("\t\t\t Classification report: \n", classification_report(y_test,y_pred))


**Dependency hand-off:** Logistic Regression is the model carried forward into financial risk
scoring and SHAP explainability (see the model comparison below for why). Saving it now.


In [ ]:
joblib.dump(model, 'models/logistic_regression_model.pkl')
print("Saved trained model -> models/logistic_regression_model.pkl")


## 5. Model 2 — Random Forest (comparison baseline)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators = 100,
    random_state = 42
)

rf_model.fit(X_train, y_train)


In [ ]:
rf_pred = rf_model.predict(X_test)

rf_pred



In [ ]:
rf_prob = rf_model.predict_proba(X_test)

rf_prob[:5]


In [ ]:
from sklearn.metrics import accuracy_score

rf_accuracy = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:")
print(rf_accuracy)


In [ ]:
from sklearn.metrics import confusion_matrix

rf_cm = confusion_matrix(y_test, rf_pred)

print("Random Forest Confustion Matrix:")
print(rf_cm)


In [ ]:
from sklearn.metrics import classification_report

print("Classification Report:")
print(classification_report(y_test, rf_pred))


## 6. Model Comparison

Logistic Regression is selected as the final model going forward — it has comparable accuracy
to Random Forest but substantially better recall on the minority ("left the company") class,
which matters more for an early-warning HR use case than overall accuracy.


In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Accuracy','Precision','Recall','F1-score'],
    'Logistic Regression': [87.76, 55.00, 44.00, 49.00],
    'Random Forest': [87.41, 67.00, 10.00, 18.00]
})

comparison


In [ ]:
comparison.to_csv(
    'results/model_comparison.csv',
    index=False)


## 7. Financial Risk Engineering

With predictions in hand, each employee's `Probability_of_Leaving` is converted into a dollar
figure: an estimated replacement cost (1.5x annual salary, a commonly used industry rule of
thumb), multiplied by the probability of leaving to get an expected `FinancialRisk`.


In [ ]:
predictions['EstimatedAnnualSalary'] = predictions['EstimatedMonthlySalary'] * 12

predictions[['EstimatedAnnualSalary','EstimatedMonthlySalary']].head()


In [ ]:
predictions['ReplacementCost'] = predictions['EstimatedAnnualSalary'] * 1.5

predictions[['EstimatedAnnualSalary','ReplacementCost']].head()


In [ ]:
predictions['FinancialRisk'] = (
    predictions['ReplacementCost'] *
    predictions['Probability_of_Leaving']
).round(2)

predictions[
    [
        'Probability_of_Leaving',
        'ReplacementCost',
        'FinancialRisk'
    ]
].head()


In [ ]:
predictions['RiskCategory'] = pd.qcut(
    predictions['FinancialRisk'],
    q = 3,
    labels = ['Low','Medium','High']
)

predictions[['FinancialRisk','RiskCategory']].head()


In [ ]:
predictions['RiskCategory'].value_counts()


In [ ]:
from babel.numbers import format_currency


In [ ]:
total_risk = predictions['FinancialRisk'].sum()

print(
    "Total Financial Risk:",
    format_currency(total_risk, 'INR', locale='en_IN')
)


In [ ]:
average_risk = predictions['FinancialRisk'].mean()

print('Average Financail Risk Per Employee: ',format_currency(average_risk, 'INR', locale='en_IN'))


In [ ]:
top10 = predictions.sort_values(
    by = 'FinancialRisk',
    ascending = False
)

top10[[
    'EstimatedAnnualSalary',
    'Probability_of_Leaving',
    'ReplacementCost',
    'FinancialRisk',
    'RiskCategory'
]].head()


In [ ]:
predictions.columns.tolist()


In [ ]:
predictions.rename(columns=
                   {
                       'Actual' : 'ActualAttrition',
                       'Predicted' : 'PredictedAttrition'
                   }, inplace = True
                   )


In [ ]:
predictions.index.equals(
    df_clean.loc[X_test.index].index
)


The original categorical columns (dropped before one-hot encoding) are rejoined from
`df_clean` for readability in the final dataset and downstream dashboards.


In [ ]:
original_columns = [
    'Attrition',
    'BusinessTravel',
    'Department',
    'EducationField',
    'Gender',
    'JobRole',
    'MaritalStatus',
    'OverTime'
]

predictions = predictions.join(
    df_clean.loc[predictions.index, original_columns]
)


In [ ]:
predictions[
    [
        'Department',
        'JobRole',
        'Gender',
        'OverTime',
        'BusinessTravel',
        'Attrition'
    ]
].head()


In [ ]:
predictions.isnull().sum()


In [ ]:
predictions.shape


In [ ]:
predictions[
    [
        'ReplacementCost',
        'Probability_of_Leaving',
        'FinancialRisk'
    ]
].describe().round(2)


In [ ]:
predictions['RiskCategory'].value_counts()


Columns are reordered into a logical, presentation-ready order before saving the final
dashboard dataset.


In [ ]:
column_order = [

    # ---------------------------
    # Employee Information
    # ---------------------------
    'Age',
    'Gender',
    'Department',
    'JobRole',
    'Education',
    'EducationField',
    'BusinessTravel',
    'MaritalStatus',
    'OverTime',

    # ---------------------------
    # Work & Experience
    # ---------------------------
    'DistanceFromHome',
    'JobLevel',
    'JobInvolvement',
    'JobSatisfaction',
    'EnvironmentSatisfaction',
    'RelationshipSatisfaction',
    'WorkLifeBalance',
    'NumCompaniesWorked',
    'TotalWorkingYears',
    'YearsAtCompany',
    'YearsInCurrentRole',
    'YearsSinceLastPromotion',
    'YearsWithCurrManager',
    'TrainingTimesLastYear',

    # ---------------------------
    # Compensation
    # ---------------------------
    'DailyRate',
    'HourlyRate',
    'MonthlyRate',
    'PercentSalaryHike',
    'PerformanceRating',
    'StockOptionLevel',
    'EstimatedMonthlySalary',
    'EstimatedAnnualSalary',
    'ReplacementCost',

    # ---------------------------
    # Prediction Results
    # ---------------------------
    'Attrition',
    'ActualAttrition',
    'PredictedAttrition',
    'Probability_of_Leaving',

    # ---------------------------
    # Financial Analysis
    # ---------------------------
    'FinancialRisk',
    'RiskCategory',

    # ---------------------------
    # Machine Learning Features
    # ---------------------------
    'BusinessTravel_Travel_Frequently',
    'BusinessTravel_Travel_Rarely',
    'Department_Research & Development',
    'Department_Sales',
    'EducationField_Life Sciences',
    'EducationField_Marketing',
    'EducationField_Medical',
    'EducationField_Other',
    'EducationField_Technical Degree',
    'Gender_Male',
    'JobRole_Human Resources',
    'JobRole_Laboratory Technician',
    'JobRole_Manager',
    'JobRole_Manufacturing Director',
    'JobRole_Research Director',
    'JobRole_Research Scientist',
    'JobRole_Sales Executive',
    'JobRole_Sales Representative',
    'MaritalStatus_Married',
    'MaritalStatus_Single',
    'OverTime_Yes'
]

predictions = predictions[column_order]

predictions.head()


**Dependency hand-off:** this is the dataset Notebook 3 loads for both the SHAP explainability
dashboard and the Power BI exports.


In [ ]:
predictions.to_csv(
    'results/predictions_with_financial_risk.csv',
    index=False)

print("Final Dashboard Dataset Saved Successfully")


## 8. Financial Risk by Business Segment

Same `predictions` dataset, sliced by department, job role, salary group, overtime, age group,
and job satisfaction to see where financial risk concentrates.


### 8.1 By Department

In [ ]:
department_risk = (
    predictions.groupby('Department')['FinancialRisk']
    .sum()
    .sort_values(ascending=False)
    .round(2)
)

department_risk


In [ ]:
ax = department_risk.plot(
    kind = 'bar',
    figsize= (8,4),
    color = [
    '#1F4E79',
    '#5B9BD5',
    '#D9EAF7'
    ]
)

for container in ax.containers:
  ax.bar_label(
      container,
      labels=[f'₹{x:,}' for x in container.datavalues]
      )

plt.title("Total Financial Risk by Department")
plt.xlabel("Department")
plt.ylabel("Financial Risk (₹)")

plt.xticks(rotation=0)

plt.tight_layout()
plt.show()


### 8.2 By Job Role

In [ ]:
jobrole_risk = (
    predictions.groupby('JobRole')['FinancialRisk']
    .sum()
    .sort_values(ascending = True)
    .round(2)
)

jobrole_risk


In [ ]:
ax = jobrole_risk.plot(
    kind = 'barh',
    figsize = (8,4),
    color = [
    '#E1BEE7',
    '#CE93D8',
    '#BA68C8',
    '#AB47BC',
    '#8E24AA',
    '#7B1FA2',
    '#6A1B9A',
    '#4A148C',
    '#2E0854'
    ]
  )

for container in ax.containers:
  ax.bar_label(
      container,
      labels = [f'₹{x:,}' for x in container.datavalues]
      )

  plt.title('Total Financial Risk By Job Role')
  plt.xlabel('Financial Risk')
  plt.ylabel('Job Role')

plt.show()



### 8.3 By Salary Group

In [ ]:
if 'SalaryGroup' not in predictions.columns:
  predictions = predictions.join(df_clean['SalaryGroup'])


In [ ]:
salary_risk = (
    predictions.groupby('SalaryGroup', observed = False)['FinancialRisk']
    .sum()
    .sort_values(ascending=False)
    .round(2)
)

salary_risk


In [ ]:
ax = salary_risk.plot(
    kind = 'bar',
    figsize = [8,4],
    color = [
    '#7C4700',
    '#B36B00',
    '#E69500',
    '#FFD166'
    ]
)

for container in ax.containers:
  ax.bar_label(
      container,
      labels = [f'₹{x:,}' for x in container.datavalues]
  )

  plt.xlabel('Salary Group')
  plt.ylabel('Financial Risk')
  plt.title('Total Financial Risk by Salary Group')

plt.xticks(rotation = 0)

plt.show()


### 8.4 By Overtime

In [ ]:
overtime_risk = (
    predictions.groupby('OverTime')['FinancialRisk']
    .sum()
    .sort_values(ascending = True)
    .round(2)
)

overtime_risk


In [ ]:
plt.figure(figsize = (6,2))

plt.hlines(
    overtime_risk.index,
    0,
    overtime_risk.values,
    color='gray'
)

plt.scatter(
    overtime_risk.values,
    overtime_risk.index,
    color = ['grey','red'],
    s=150
)

for i, value in enumerate(overtime_risk.values):
  plt.text(value, i, f'₹{value:,.0f}', va ='center')

plt.grid(axis = 'x', linestyle = '--', alpha = 0.3)

plt.title('Total Financial Risk by Overtime')
plt.xlabel('Financial Risk (₹)')
plt.ylabel('OverTime')

plt.show()


### 8.5 By Age Group

In [ ]:
predictions = predictions.join(df_clean['AgeGroup'])


In [ ]:
age_risk = (
    predictions.groupby('AgeGroup',observed=False)['FinancialRisk']
    .sum()
    .sort_values()
    .round(2)
)

age_risk


In [ ]:
ax = age_risk.plot(
    kind='line',
    marker='o',
    linewidth='3',
    markersize='8',
    figsize=(8,4),
    color='#7B1FA2'
)

for x, y in enumerate(age_risk.values):
  plt.text(x,y, f'₹{y:,.0f}')

  plt.title('Total Financial Risk By Age Group')
  plt.xlabel('Age Group')
  plt.ylabel('Financial Risk')

plt.ticklabel_format(style='plain', axis='y')

plt.grid(axis='y', linestyle='--', alpha = 0.3)


### 8.6 By Job Satisfaction

In [ ]:
job_sat_risk = (
    predictions.groupby('JobSatisfaction')['FinancialRisk']
    .sum()
    .round(2)
)

job_sat_risk


In [ ]:
ax = job_sat_risk.plot(
    kind='line',
    linewidth=0,
    marker='o',
    markersize=0
)

plt.fill_between(
    job_sat_risk.index,
    job_sat_risk.values,
    color='#009688',
    alpha=0.3
)

for x, y in zip(job_sat_risk.index, job_sat_risk.values):
    plt.text(
        x,
        y + 200000,
        f'₹{y:,.0f}',
        ha='center'
    )

plt.title('Total Financial Risk by Job Satisfaction')
plt.xlabel('Job Satisfaction')
plt.ylabel('Financial Risk')

plt.ticklabel_format(style='plain', axis='y')
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.show()


**Key findings from segment analysis:** R&D contributes the highest total financial risk of
any department; Sales Executives carry the highest risk by job role; the High salary band
(not the highest-paid group) contributes the most total risk because it combines a large
headcount with meaningful attrition probability; overtime workers and the 26–45 age range
dominate financial risk; and mid-range job satisfaction (level 3) contributes more risk than
either extreme.


---
**Next:** open `03_explainability_and_dashboards.ipynb` to explain individual predictions with
SHAP and produce the Power BI-ready exports. That notebook reloads the model, scaler, splits,
and `predictions_with_financial_risk.csv` saved above — it does not depend on this kernel still
being open.
